In [39]:
model = "llama3.2:1B"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [ ]:
from langchain_ollama import ChatOllama

model_name = "llama3.2:1B"

# ADD HERE YOUR CODE
model = ChatOllama(model=model_name)

In [41]:
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
# Stellt sicher, dass 'bge-m3' in Ollama gepullt wurde
embedding_model = OllamaEmbeddings(model="bge-m3")
# ADD HERE YOUR CODE
# Stellt sicher, dass 'bge-m3' in Ollama gepullt wurde
embedding_model = OllamaEmbeddings(model="bge-m3")

In [42]:
from langchain_chroma import Chroma
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_or_create_collection(name="rag_collection")

# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(
    client=client,
    collection_name="rag_collection",
    embedding_function=embedding_model,
)

In [43]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)

# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ADD HERE YOUR CODE
chain = prompt | model | StrOutputParser()

In [ ]:
#neuer block für Datenbank
        
from langchain_core.documents import Document

# 1. Beispieldaten erstellen
data = [
    Document(page_content="Supervised learning uses labeled data to train models.", metadata={"category": "ML"}),
    Document(page_content="Unsupervised learning finds hidden patterns in unlabeled data.", metadata={"category": "ML"}),
    Document(page_content="Reinforcement learning involves an agent learning through rewards.", metadata={"category": "ML"})
]

# 2. Die Daten in die Datenbank schreiben
vector_db_from_client.add_documents(data)
print("Daten wurden erfolgreich in Chroma gespeichert.")
# ----------------------------

Daten wurden erfolgreich in Chroma gespeichert.


In [45]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query)

print(docs)

[Document(id='021208ea-0d17-45c4-9b27-af7f20a6a20f', metadata={'category': 'ML'}, page_content='Supervised learning uses labeled data to train models.'), Document(id='b4cedcf7-8b9b-451d-970a-583fce8fb64c', metadata={'category': 'ML'}, page_content='Unsupervised learning finds hidden patterns in unlabeled data.'), Document(id='55437c8d-9f0a-4ea6-af76-f16c03570e59', metadata={'category': 'ML'}, page_content='Reinforcement learning involves an agent learning through rewards.')]


In [46]:
# ADD HERE YOUR CODE
chain.invoke({"docs": format_docs(docs)})

'Based on the retrieved documents, the main themes that emerge are:\n\n1. **Training Models**: The first two points highlight the process of supervised and unsupervised machine learning, where labeled (with correct answers) and unlabeled (without labels) data is used to train models.\n2. **Pattern Discovery**: Unsupervised learning is focused on finding hidden patterns in unlabeled data, which can reveal underlying relationships or structures that might not be apparent through traditional data analysis.\n3. **Agent Learning**: Reinforcement learning involves an agent learning through rewards, where the goal is to maximize a cumulative reward function by making decisions based on feedback from the environment.\n\nThese themes suggest that machine learning techniques are used to uncover patterns, learn from data, and improve decision-making processes in various fields, including predictive modeling, data analysis, and automation.'

In [47]:
# Simple stream the chain output
# ADD HERE YOUR CODE
for chunk in chain.stream({"docs": format_docs(docs)}):
    print(chunk, end="", flush=True)

Based on the retrieved documents, the main themes are:

1. **Data Types**:
   - **Supervised Learning**: Uses labeled data to train models.
   - **Unsupervised Learning**: Finds hidden patterns in unlabeled data.
   - **Reinforcement Learning**: Involves an agent learning through rewards.

2. **Learning Process**:
   - The process of training models using data involves the interaction between algorithms and their inputs, resulting in improved model performance over time.

3. **Agent Behavior**:
   - Reinforcement learning involves an agent that learns to make decisions based on rewards or penalties received as a result of its actions.

4. **Pattern Recognition**:
   - Unsupervised learning is used to find hidden patterns or structures within the data, whereas supervised learning aims to predict specific outcomes based on labeled examples.

In [48]:
# More complex async event streaming
# ADD HERE YOUR CODE
async for event in chain.astream_events({"docs": format_docs(docs)}, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

The two documents you've retrieved appear to be from various sources, but I'll provide a summary of the main themes based on the content:

**Supervised Learning:**

* The theme is about using labeled data to train models.
* This approach assumes that there is already some amount of knowledge or information available (labeled data).
* The goal is to learn patterns and relationships from this pre-existing knowledge.

**Unsupervised Learning:**

* The theme focuses on discovering hidden patterns in unlabeled data.
* Unsupervised learning involves analyzing data without prior knowledge about the expected outcomes or patterns.
* This approach often helps identify clusters, groupings, or other structures within the data that may not be apparent with labeled data.

**Reinforcement Learning:**

* The main theme revolves around an agent (typically a computer program) learning through rewards.
* Reinforcement learning is centered on the interaction between the agent and its environment, where th

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [49]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever()

# ADD HERE YOUR CODE
qa_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

In [50]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000014073FB5BA0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"), additional_kwargs={})])
| ChatOllama(model='llama3.2:1B')
| StrOutputParser()

In [51]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
qa_rag_chain.invoke(question)

'Supervised learning is a type of machine learning that uses labeled data to train models, where the correct output or response is already known. The goal is for the model to learn a mapping between inputs and outputs, based on the existing labels. In this process, the model learns to predict the output given the input, by analyzing the patterns in the labeled data.'

In [52]:
# More complex async event streaming
# ADD HERE YOUR CODE
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where algorithms are trained on labeled data, meaning that each example in the training set has been annotated with the corresponding output or response. The goal is to learn a mapping between input features and their corresponding labels, so the algorithm can make predictions on new, unseen data. Supervised learning requires labeled data for effective model training.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [53]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [54]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [55]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [56]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where the algorithm is trained on labeled data, meaning that each example in the dataset has a corresponding output or target variable that it should predict. In other words, the training data includes both input data and corresponding labels or outputs, and the model learns to map inputs to their respective labels through supervised classification or regression.

Here's an overview of how supervised learning works:

1. The algorithm is fed into the labeled dataset.
2. It tries to find a pattern or relationship between the input data and the labels.
3. Once it identifies the patterns, it uses this knowledge to make predictions on new, unseen input data.

Supervised learning has several key characteristics:

* **Labeled data**: Each example in the dataset has a corresponding label or output.
* **Training set**: The labeled data is used to train the model.
* **Testing set**: A separate test dataset is used to evaluate the performance of t

In [57]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Here is the rephrased follow up question:

What types of machine learning algorithms are commonly applied in fields such as image recognition, speech recognition, NLP, and time series forecasting to supervised learning?I know the general answer.

Can you please specify which fields (e.g., image recognition, speech recognition) within those domains typically use supervised learning for their respective tasks? I'll do my best to provide more tailored information.